# Tiger POMDP: Learning Finite State Controllers with MAPSO, EM, and SGD

This notebook implements and benchmarks three optimization methods for learning
Finite State Controller (FSC) parameters from trajectory data generated in the
Tiger POMDP environment. The objective is to estimate the FSC parameters that
maximize the likelihood of observed action-observation sequences.

A Finite State Controller is a compact policy representation that maintains a
finite amount of internal memory. Unlike reactive policies that depend only on
the current observation, FSCs use hidden memory states to summarize past
information, making them suitable for partially observable environments.

Three optimization approaches are implemented and compared:

1. **Modified Adaptive Particle Swarm Optimization (MAPSO)** — a population-based
   global optimizer
2. **Expectation-Maximization (EM)** — a likelihood-based local optimizer
3. **Stochastic Gradient Descent (SGD)** — gradient-based local optimization via
   Fisher's identity

A central result explored here: near-zero training NLL does not imply that the
learned FSC recovers the *correct* policy. Alongside raw NLL, this notebook
also evaluates exact policy match against the ground-truth Bayesian agent, and
generalization via a held-out validation split.

## Contents

1. Ground-truth agent and dataset generation
2. Optimizer hyperparameters
3. Fit with MAPSO
4. Fit with EM
5. Fit with SGD
6. Diagnostics: convergence plots
7. Warm-started EM (initialized from the MAPSO solution)
8. Warm-started SGD (initialized from the MAPSO solution)
9. Hybrid optimization: policy-match evaluation across MAPSO checkpoints
10. Label-switching alignment and comparison tables
11. FSC visualization
12. Train/val NLL scatter across all trials
13. Persist results to `results_log.csv`

## Requirements

- Local modules: `tiger_pomdp`, `mapso`, `em`, `sgd`, `val`
- Python packages: `numpy`, `scipy`, `matplotlib`, `pandas`, `joblib`

## Reproducibility notes

- Fitting cells (MAPSO/EM/SGD) cache their results to `.npz` files under
  `*_checkpoints/` directories, keyed by `(mc, n_data, gen_seed)`. Delete the
  relevant cache file to force a re-run.
- Long-running cells (MAPSO restarts, the hybrid checkpoint sweep) may take
  significant time on first run; subsequent runs load from cache.


In [ ]:
from tiger_pomdp import *  # Tiger POMDP environment and ground-truth agent
from mapso import *        # Modified Adaptive Particle Swarm Optimization (MAPSO)
from em import *           # Expectation-Maximization (EM)
from sgd import *          # Stochastic Gradient Descent (SGD)
from val import *          # Validation-set generation and evaluation helpers


## Reachable memory complexity (`mc`) by (δ, α)

The Tiger POMDP's exact memory-state solution depends on `delta` (`δ`) and
`alpha` (`α`). Not every `(δ, α)` pair yields every target `mc`; the table
below records which `(δ, α)` combinations reach a given `mc`, found by
bisecting for `α` at fixed `δ` (see `mc_params` in the dataset-generation cell
below). Cells marked `invalid` fall outside the valid parameter range;
`not reachable` means no `α` in range attains that `mc` at that `δ`.


| mc | δ=0.2 | δ=0.35 | δ=0.5 | δ=0.65 | δ=0.8 |
|----|----|----|----|----|----|
| 1 | invalid (α>1) | invalid (α>1) | invalid (α>1) | invalid (α>1) | 0.7737 |
| 2 | invalid (α>1) | 0.7093 | 0.422 | 0.2456 | 0.118 |
| 3 | 0.7721 | 0.355 | 0.1678 | 0.06605 | 0.0161 |
| 4 | 0.5072 | 0.1924 | 0.06518 | 0.01566 | 0.001871 |
| 5 | 0.3524 | 0.1038 | 0.02372 | 0.003448 | 0.0002094 |
| 6 | 0.2502 | 0.0544 | 0.008242 | 0.0007392 | 2.329e-05 |
| 7 | 0.1784 | 0.02773 | 0.002797 | 0.0001572 | 2.588e-06|
| 8 | 0.1268 | 0.01382 | 0.0009391 | 3.338e-05 | not reachable |

## 1. Ground-truth agent and dataset generation

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from scipy.optimize import linear_sum_assignment

np.random.seed(0)

mc = 3
n_data = 20             # this is your TRAINING set size, not the total episodes
n_val = int(n_data)     # pick however much held-out data you want on top of that

# Map (mc, n_data) tuples directly to seed values
SEED_MAP = {(2, 5): 22,(2, 10): 22,(4, 15): 22,(5, 25): 22,(5, 30): 22}

# Fetch the seed, falling back to a default (e.g., 0) if the pair isn't listed
gen_seed = SEED_MAP.get((mc, n_data), 1)

mc_params = {
    1: {"delta": 0.85, "alpha": 0.3},
    2: {"delta": 0.5,  "alpha": 0.2},
    3: {"delta": 0.5,  "alpha": 0.1},
    4: {"delta": 0.5,  "alpha": 0.05},
    5: {"delta": 0.5,  "alpha": 0.024},
    6: {"delta": 0.2,  "alpha": 0.2502},
    7: {"delta": 0.2,  "alpha": 0.1784},
    8: {"delta": 0.2,  "alpha": 0.1268},
}

params = mc_params[mc]
pomdp = TigerPOMDP(delta=params["delta"], alpha=params["alpha"])


V, mc = solve_memory_mdp(pomdp)
fsc = build_memory_fsc(pomdp, mc)
ms_sorted = sorted(fsc.keys())
M = len(ms_sorted)
m_to_idx = {m: i for i, m in enumerate(ms_sorted)}
A, Y = 3, 2
pomdp.action_names = {-1: "OpenLeft", 0: "Listen", 1: "OpenRight"}
pomdp.obs_names = {-1: "HearLeft", 1: "HearRight"}
print(f"Exact agent: mc={mc}, nodes={ms_sorted}")

# Generate train and val as TWO INDEPENDENT calls to the same generator,
# using the exact same generative process/config (pomdp, fsc, m0, steps) --
# val is now treated exactly like train (same function, same args, same
# encode_dataset step), just drawn fresh with a different seed instead of
# being carved out of one shared pool via index-splitting. Splitting a
# single small batch (n_data + n_val = 20 episodes total) meant train/val
# were negatively-correlated finite samples from each other, which is a
# real source of "val NLL < train NLL" noise at this sample size -- an
# independent fresh draw removes that coupling.
val_gen_seed = gen_seed + 10_000  # disjoint from gen_seed, so val can't overlap train

dataset_raw = generate_dataset(pomdp, fsc, n_episodes=n_data, m0=0, steps=20, seed=gen_seed)
dataset_encoded = encode_dataset(dataset_raw)
train_set = dataset_encoded

val_raw = generate_dataset(pomdp, fsc, n_episodes=n_val, m0=0, steps=20, seed=val_gen_seed)
val_set = encode_dataset(val_raw)

print(f"train: {len(dataset_encoded)}, val: {len(val_set)}")

# Ground truth NLL evaluation
theta_true, _, _, _ = build_ground_truth_theta(fsc, m0=0, big=25.0)
rho_true, pi_true, g_true = unpack_theta(theta_true, M, A, Y)
params = (dataset_encoded, M, A, Y)
nll_true, _ = tiger_negative_loglik(theta_true, params)
print(f"Ground-truth NLL (train): {nll_true:.3e}\n")

# Print the first 5 trajectories from the training set
for idx, episode in enumerate(dataset_raw[:5]):
    # Extract the individual sequences from the list of step dictionaries
    memories = [step['m'] for step in episode['trajectory']]
    actions = [step['a'] for step in episode['trajectory']]
    observations = [step['z'] for step in episode['trajectory']]
    
    print(f"--- Trajectory {idx + 1} (True State: {episode['true_state']}) ---")
    print(f"Memories:     {memories}")
    print(f"Actions:      {actions}")
    print(f"Observations: {observations}")
    print()

## 2. Optimizer hyperparameters

Shared restart/iteration budget used by MAPSO, EM, and SGD below.

In [ ]:
n_particles = 200
n_iterations = 2000
n_restarts=50

### 2.1 Fit with MAPSO

In [ ]:
import time

print("--- MAPSO ---")


def build_trainable_mask(M, A, Y, listen_idx=1):
    dim_rho, dim_pi, dim_g = M, A * M, M * M * A * Y
    total = dim_rho + dim_pi + dim_g
    mask = np.ones(total, dtype=bool)
    idx = dim_rho + dim_pi
    for mp in range(M):
        for m in range(M):
            for a in range(A):
                for y in range(Y):
                    if a != listen_idx:
                        mask[idx] = False
                    idx += 1
    return mask


dimension = M * (1 + A + M * A * Y)
mask = build_trainable_mask(M, A, Y, listen_idx=1)

start_time = time.perf_counter()

best_gbest, best_gbest_value, best_restart, results, all_nll, all_nll_histories = run_optimization_with_restarts_checkpointed(
    tiger_negative_loglik,
    params,
    trainable_mask=mask,
    n_dimensions=dimension,
    n_particles=n_particles,
    n_iterations=n_iterations,
    num_neighbors_init=3,
    num_neighbors_final=100,
    num_neighbors_mid=30,
    n_restarts=n_restarts,
    seed=0,
    checkpoint_dir="./mapso_checkpoints",
    run_id=f"mc{mc}_n{n_data}_seed{gen_seed}",
    checkpoint_every=200,
    patience=50,
)

end_time = time.perf_counter()
mapso_runtime = end_time - start_time

mapso_history = all_nll_histories[best_restart]

theta_mapso = best_gbest
rho_mapso, pi_mapso, g_mapso = unpack_theta(theta_mapso, M, A, Y)

theta_per_restart_mapso = [r["gbest"] for r in results]
mapso_trials = [unpack_theta(theta, M, A, Y) for theta in theta_per_restart_mapso]

theta_history = results[best_restart]["theta_history"]

n_checkpoints=30
checkpoint_iters = np.unique(
    np.round(
        np.geomspace(1, len(theta_history), n_checkpoints)
    ).astype(int) - 1
)

print(f"NLL across restarts: min={all_nll.min():.3e}, max={all_nll.max():.3e}, std={all_nll.std():.3e}")
print(f"Running time: {mapso_runtime:.4f} seconds")
print(f"Using {len(checkpoint_iters)} MAPSO checkpoints:")
print(checkpoint_iters)

### 2.2 Fit with EM

Results are cached to `.npz`; delete the corresponding cache file under `em_checkpoints/` to force a re-run.

In [ ]:
import os
import time
import numpy as np

print("--- EM ---")

cache_file = f"em_checkpoints/mc{mc}_n{n_data}_seed{gen_seed}.npz"

if os.path.exists(cache_file):
    print(f"Loading saved results from {cache_file}...")
    with np.load(cache_file, allow_pickle=True) as data:
        rho_em = data['rho_em']
        pi_em = data['pi_em']
        g_em = data['g_em']
        em_nll = data['em_nll'].item()
        em_best_restart = data['em_best_restart'].item()
        em_results = data['em_results'].tolist() 
        em_all_nll = data['em_all_nll']
        em_all_nll_histories = data['em_all_nll_histories'].tolist()
        em_runtime = data['em_runtime'].item()
        
    # Print the NLL for every restart loaded from the cache
    print(f"Loaded {len(em_all_nll)} restarts:")
    for i, nll in enumerate(em_all_nll):
        marker = " (Best)" if i == em_best_restart else ""
        print(f"  Restart {i+1}/{len(em_all_nll)} - NLL: {nll:.3e}{marker}")

else:
    print("Running EM... (this may take a while)")
    start_time = time.perf_counter()

    rho_em, pi_em, g_em, em_nll, em_best_restart, em_results, em_all_nll, em_all_nll_histories = train_em_with_restarts_parallel(
        dataset_encoded, M, A, Y,
        max_iter=n_iterations, n_restarts=n_restarts, seed=0,
    )

    end_time = time.perf_counter()
    em_runtime = end_time - start_time
    
    # Cast variable-length or complex structures to dtype=object
    np.savez(cache_file,
             rho_em=rho_em,
             pi_em=pi_em,
             g_em=g_em,
             em_nll=em_nll,
             em_best_restart=em_best_restart,
             em_results=np.array(em_results, dtype=object),
             em_all_nll=em_all_nll,
             em_all_nll_histories=np.array(em_all_nll_histories, dtype=object),
             em_runtime=em_runtime)
             
    print(f"Results saved to {cache_file}.")

em_nll_hist = em_all_nll_histories[em_best_restart]
em_trials = [(r['rho'], r['pi'], r['g']) for r in em_results]
print(f"\nEM final NLL: {em_nll_hist[-1]:.3e}")
print(f"Running time: {em_runtime:.4f} seconds")

### 2.3 Fit with SGD

`sgd.py`'s bundled `train_sgd_restarts_parallel` unpacks `_run_one_restart`'s output as a 7-tuple, but `_run_one_restart` actually returns 8 values (it includes `is_warm`). Rather than edit `sgd.py`, the fixed version is redefined here and shadows the imported name — everything else in `sgd.py` is used unchanged. Results are cached to `.npz` under `sgd_checkpoints/`.

In [ ]:
import os
import time
import numpy as np
import sgd as _sgd
from joblib import Parallel, delayed


def train_sgd_restarts_parallel(dataset, M, A=3, Y=2, n_restarts=5, seed=0,
                                 n_jobs=-1, warm_params=None, warm_restarts=None,
                                 warm_sigma=0.1, **sgd_kwargs):
    tasks = []
    first_warm_seen = False
    for r in range(n_restarts):
        seed_r = seed + r
        is_warm = warm_params is not None and ((warm_restarts is None) or (r in warm_restarts))
        if is_warm:
            rho_warm, pi_warm, g_warm = warm_params
            jitter = 0.0 if not first_warm_seen else warm_sigma
            first_warm_seen = True
            tasks.append((seed_r, rho_warm, pi_warm, g_warm, jitter, True))
        else:
            tasks.append((seed_r, None, None, None, 0.0, False))

    raw_results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_sgd._run_one_restart)(seed_r, dataset, M, A, Y, sgd_kwargs,
                                        init_rho=ir, init_pi=ip, init_g=ig,
                                        init_jitter=ij, is_warm=iw)
        for seed_r, ir, ip, ig, ij, iw in tasks
    )

    all_nll = np.zeros(n_restarts)
    all_nll_histories = []
    results = []
    best_nll = np.inf
    best_restart = -1
    best = None

    for i, r in enumerate(raw_results):
        final_nll, rho, pi, g, nll_history, time_history, seed_r, is_warm = r
        results.append({
            'restart': i, 'seed': seed_r, 'warm_started': is_warm,
            'rho': rho, 'pi': pi, 'g': g,
            'nll_value': final_nll, 'nll_history': nll_history,
            'time_history': time_history,
        })
        all_nll[i] = final_nll
        all_nll_histories.append(nll_history)

        tag = ' [warm]' if is_warm else ''
        print(f'[restart {i}] seed={seed_r}  final NLL={final_nll:.6f}{tag}')

        if final_nll < best_nll:
            best_nll = final_nll
            best_restart = i
            best = (rho, pi, g, nll_history, time_history, seed_r)

    print(f'\nBest restart: {best_restart}  (NLL={best_nll:.6f})')
    best_rho, best_pi, best_g, best_nll_history, best_time_history, best_seed = best
    return (best_rho, best_pi, best_g, best_nll_history, best_time_history, best_seed,
            best_restart, results, all_nll, all_nll_histories)


print("--- SGD ---")

cache_file = f"sgd_checkpoints/mc{mc}_n{n_data}_seed{gen_seed}.npz"

if os.path.exists(cache_file):
    print(f"Loading saved results from {cache_file}...")
    with np.load(cache_file, allow_pickle=True) as data:
        rho_sgd = data['rho_sgd']
        pi_sgd = data['pi_sgd']
        g_sgd = data['g_sgd']
        sgd_nll_hist = data['sgd_nll_hist']
        sgd_time_history = data['sgd_time_history']
        sgd_best_seed = data['sgd_best_seed'].item()
        sgd_best_restart = data['sgd_best_restart'].item()
        sgd_results = data['sgd_results'].tolist()
        sgd_all_nll = data['sgd_all_nll']
        sgd_all_nll_histories = data['sgd_all_nll_histories'].tolist()
        sgd_runtime = data['sgd_runtime'].item()

    # Print the NLL for every restart loaded from the cache
    print(f"Loaded {len(sgd_all_nll)} restarts:")
    for i, nll in enumerate(sgd_all_nll):
        marker = " (Best)" if i == sgd_best_restart else ""
        print(f"  Restart {i+1}/{len(sgd_all_nll)} - NLL: {nll:.3e}{marker}")

else:
    print("Running SGD... (this may take a while)")
    start_time = time.perf_counter()
    
    (rho_sgd, pi_sgd, g_sgd, sgd_nll_hist, sgd_time_history, sgd_best_seed,
     sgd_best_restart, sgd_results, sgd_all_nll, sgd_all_nll_histories) = (
        train_sgd_restarts_parallel(dataset_encoded, M, n_epochs=n_iterations, n_restarts=n_restarts,
                                    seed=0, momentum=0.5, patience=100, tol=1e-12, lr=2,
                                    dataset_nll_fn=dataset_nll)   # <-- add this
    )

    end_time = time.perf_counter()
    sgd_runtime = end_time - start_time

    # Ensure the directory exists before saving
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)

    # Cast variable-length or complex structures to dtype=object
    np.savez(cache_file,
             rho_sgd=rho_sgd,
             pi_sgd=pi_sgd,
             g_sgd=g_sgd,
             sgd_nll_hist=sgd_nll_hist,
             sgd_time_history=sgd_time_history,
             sgd_best_seed=sgd_best_seed,
             sgd_best_restart=sgd_best_restart,
             sgd_results=np.array(sgd_results, dtype=object),
             sgd_all_nll=sgd_all_nll,
             sgd_all_nll_histories=np.array(sgd_all_nll_histories, dtype=object),
             sgd_runtime=sgd_runtime)

    print(f"Results saved to {cache_file}.")

sgd_trials = [(r['rho'], r['pi'], r['g']) for r in sgd_results]
print(f"\nSGD final NLL: {sgd_nll_hist[-1]:.3e}")
print(f"Running time: {sgd_runtime:.4f} seconds")

## 3. Diagnostics: convergence plots

Final NLL per restart and convergence curves for all three methods, best restart highlighted.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

def enforce_monotonic_best(history):
    '''Post-hoc fix for chunk-boundary resets: collapse any upward
    jump into a flat plateau at the last true best value.'''
    return np.minimum.accumulate(np.asarray(history))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {'MAPSO': 'tab:blue', 'EM': 'tab:orange', 'SGD': 'tab:green'}

# --- Left: final NLL per restart, all three algorithms ---
ax = axes[0]
ax.scatter(range(len(all_nll)), all_nll, label='MAPSO', color=colors['MAPSO'])
ax.axhline(all_nll.min(), color=colors['MAPSO'], linestyle='--', alpha=0.5)

ax.scatter(range(len(em_all_nll)), em_all_nll, label='EM', color=colors['EM'])
ax.axhline(em_all_nll.min(), color=colors['EM'], linestyle='--', alpha=0.5)

ax.scatter(range(len(sgd_all_nll)), sgd_all_nll, label='SGD', color=colors['SGD'])
ax.axhline(sgd_all_nll.min(), color=colors['SGD'], linestyle='--', alpha=0.5)

ax.set_xlabel('Restart')
ax.set_ylabel('Final NLL')
ax.set_title('Final NLL across restarts')
ax.legend()

# --- Right: convergence curves, all restarts overlaid, best highlighted ---
ax = axes[1]

all_nll_histories = [enforce_monotonic_best(c) for c in all_nll_histories]
for i in range(len(all_nll_histories)):
    ax.plot(all_nll_histories[i], color=colors['MAPSO'], alpha=0.15)
ax.plot(all_nll_histories[best_restart], color=colors['MAPSO'], linewidth=2, label='MAPSO (best)')

for i in range(len(em_all_nll_histories)):
    ax.plot(em_all_nll_histories[i], color=colors['EM'], alpha=0.15)
ax.plot(em_all_nll_histories[em_best_restart], color=colors['EM'], linewidth=2, label='EM (best)')

for i in range(len(sgd_all_nll_histories)):
    ax.plot(sgd_all_nll_histories[i], color=colors['SGD'], alpha=0.15)
ax.plot(sgd_all_nll_histories[sgd_best_restart], color=colors['SGD'], linewidth=2, label='SGD (best)')

ax.set_xlabel('Iteration / epoch')
ax.set_ylabel('NLL')
ax.set_title('Convergence curves across restarts')
ax.legend()

plt.tight_layout()
#plt.show()

## 4. Warm-started EM (MAPSO initialization)

EM re-run with initial parameters set to the best MAPSO solution, to test whether warm-starting from a good global-search basin improves on EM's random-restart best.

In [ ]:
import os, time
import numpy as np

cache_file=f"em_warm_checkpoints/mc{mc}_n{n_data}_seed{gen_seed}.npz"

if os.path.exists(cache_file):
    print(f"Loading EM (MAPSO warm-start) from {cache_file}...")
    with np.load(cache_file, allow_pickle=True) as data:
        rho_em_warm=data['rho_em_warm']
        pi_em_warm=data['pi_em_warm']
        g_em_warm=data['g_em_warm']
        em_warm_nll_hist=data['em_warm_nll_hist']
        em_warm_runtime=data['em_warm_runtime'].item()
else:
    print("Running EM with MAPSO warm-start...")
    rho_mapso, pi_mapso, g_mapso=unpack_theta(best_gbest, M, A, Y)
    start_time=time.perf_counter()
    
    rho_em_warm, pi_em_warm, g_em_warm, _, em_warm_nll_hist, _=train_em(
        dataset_encoded, M, A, Y, max_iter=2000, verbose=False,
        init_rho=rho_mapso, init_pi=pi_mapso, init_g=g_mapso, tol=0
    )
    
    em_warm_runtime=time.perf_counter() - start_time
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)
    np.savez(cache_file, rho_em_warm=rho_em_warm, pi_em_warm=pi_em_warm, g_em_warm=g_em_warm, 
             em_warm_nll_hist=em_warm_nll_hist, em_warm_runtime=em_warm_runtime)
    print(f"Results saved to {cache_file}.")

print(f"EM (MAPSO warm-start) final NLL: {em_warm_nll_hist[-1]:.3e}\nEM (random restarts) best NLL: {em_nll:.3e}")

## 5. Warm-started SGD (MAPSO initialization)

Same idea as the warm-started EM above, applied to SGD.

In [ ]:
import os, time
import numpy as np

cache_file=f"sgd_warm_checkpoints/mc{mc}_n{n_data}_seed{gen_seed}.npz"

if os.path.exists(cache_file):
    print(f"Loading Multi SGD (MAPSO warm-start) from {cache_file}...")
    with np.load(cache_file, allow_pickle=True) as data:
        sgd_warm_best_rho=data['sgd_warm_best_rho']
        sgd_warm_best_pi=data['sgd_warm_best_pi']
        sgd_warm_best_g=data['sgd_warm_best_g']
        sgd_warm_best_nll_history=data['sgd_warm_best_nll_history']
        sgd_warm_best_time_history=data['sgd_warm_best_time_history']
        sgd_warm_best_seed=data['sgd_warm_best_seed'].item()
        sgd_warm_best_restart=data['sgd_warm_best_restart'].item()
        sgd_warm_results=data['sgd_warm_results'].tolist()
        sgd_warm_all_nll=data['sgd_warm_all_nll']
        sgd_warm_all_nll_histories=data['sgd_warm_all_nll_histories'].tolist()
        sgd_warm_runtime=data['sgd_warm_runtime'].item()
else:
    print("Running Multi SGD with MAPSO warm-start...")
    start_time=time.perf_counter()
    
    (sgd_warm_best_rho, sgd_warm_best_pi, sgd_warm_best_g, sgd_warm_best_nll_history,
     sgd_warm_best_time_history, sgd_warm_best_seed, sgd_warm_best_restart,
     sgd_warm_results, sgd_warm_all_nll, sgd_warm_all_nll_histories)=train_sgd_restarts_parallel(
        dataset_encoded, M, A, Y, n_restarts=8, seed=0,
        warm_params=(rho_mapso, pi_mapso, g_mapso), warm_restarts=None, warm_sigma=0.05, 
        lr=2.0, n_epochs=200, tol=1e-12, momentum=0.5, patience=100
    )
    
    sgd_warm_runtime=time.perf_counter() - start_time
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)
    
    np.savez(cache_file, sgd_warm_best_rho=sgd_warm_best_rho, sgd_warm_best_pi=sgd_warm_best_pi,
             sgd_warm_best_g=sgd_warm_best_g, sgd_warm_best_nll_history=sgd_warm_best_nll_history,
             sgd_warm_best_time_history=sgd_warm_best_time_history, sgd_warm_best_seed=sgd_warm_best_seed,
             sgd_warm_best_restart=sgd_warm_best_restart, sgd_warm_results=np.array(sgd_warm_results, dtype=object),
             sgd_warm_all_nll=sgd_warm_all_nll, sgd_warm_all_nll_histories=np.array(sgd_warm_all_nll_histories, dtype=object),
             sgd_warm_runtime=sgd_warm_runtime)
    print(f"Results saved to {cache_file}.")

print(f"Multi SGD (MAPSO warm-start):\nBest restart: {sgd_warm_best_restart} (seed={sgd_warm_best_seed})")
print(f"Best final NLL: {sgd_warm_best_nll_history[-1]:.1e}")
print(f"NLL spread: min={sgd_warm_all_nll.min():.1e}, max={sgd_warm_all_nll.max():.1e}, std={sgd_warm_all_nll.std():.1e}")

## 6. Hybrid optimization: policy-match evaluation across MAPSO checkpoints

For a range of MAPSO iteration checkpoints (log-spaced), warm-start EM and SGD
from that checkpoint's parameters and evaluate:

- Final NLL for MAPSO (at that checkpoint), EM, and SGD
- **Policy match**: whether the learned model's greedy policy exactly
  reproduces the true Bayesian agent's action choices when replayed on its own
  internal state trajectory (`replay_policy_match`), evaluated on the held-out
  validation set

This tests whether letting MAPSO explore only briefly before handing off to a
local optimizer is enough to reach the correct policy, not just a low NLL.
Results and full NLL histories are cached to JSON under `hybrid_checkpoints/`.

In [ ]:
import os, time, json
import numpy as np

# --------------------------------------------------------------------
# Policy-match test: does a learned (rho, pi, g) reproduce the true
# Bayesian-agent's *hard* action choices, replayed on its own internal
# state trajectory? No cross-mc / cross-labeling alignment required --
# we only ever follow the learned model's own node index forward.
# --------------------------------------------------------------------
def replay_policy_match(rho, pi, g, dataset_encoded):
    """
    rho: (M,)      pi: (A, M)      g: (M, M, A, Y), g[mp, m, a, y] = P(next=mp | m,a,y)
    dataset_encoded: list of (actions, obs) int-sequences, same format
                      passed to dataset_nll / train_em / train_sgd.

    Returns (step_agreement, traj_agreement):
      step_agreement : fraction of (trajectory, timestep) pairs where the
                        learned model's argmax action matches the true action
      traj_agreement : fraction of whole trajectories matched at every step
    """
    m0 = int(np.argmax(rho))
    total_steps = matched_steps = 0
    matched_traj = 0
    n_traj = len(dataset_encoded)

    for actions, obs in dataset_encoded:
        m = m0
        traj_ok = True
        T = len(actions)
        for t in range(T):
            a_pred = int(np.argmax(pi[:, m]))
            a_true = int(actions[t])
            total_steps += 1
            if a_pred == a_true:
                matched_steps += 1
            else:
                traj_ok = False
            if t < T - 1:
                y_true = int(obs[t])
                m = int(np.argmax(g[:, m, a_true, y_true]))
        if traj_ok:
            matched_traj += 1

    return matched_steps / total_steps, matched_traj / n_traj


def policy_matched(rho, pi, g, dataset_encoded, threshold=1.0):
    step_agree, traj_agree = replay_policy_match(rho, pi, g, dataset_encoded)
    return step_agree >= threshold, step_agree, traj_agree


n_restart_hybrid = 20
cache_file = f"hybrid_checkpoints/mc{mc}_n{n_data}_seed{gen_seed}_nr{n_restart_hybrid}_nc{n_checkpoints}_policytest.json"
history_cache_file = f"hybrid_checkpoints/mc{mc}_n{n_data}_seed{gen_seed}_nr{n_restart_hybrid}_nc{n_checkpoints}_histories.json"

def numpy_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.bool_):
        return bool(obj)
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

if os.path.exists(cache_file) and os.path.exists(history_cache_file):
    print(f"Loading saved hybrid results from {cache_file}...")
    with open(cache_file, "r") as f:
        hybrid_results = json.load(f)
    with open(history_cache_file, "r") as f:
        hybrid_histories = json.load(f)  # dict keyed by str(mapso_iter) -> {"em_history": [...], "sgd_history": [...]}
    print(f"Loaded {len(hybrid_results)} checkpoints (with histories).")
    for res in hybrid_results:
        print(f"  Iter {res['mapso_iter']:>4} - "
              f"MAPSO: {res['mapso_nll']:.3e} (match={res['mapso_matched']}) | "
              f"EM: {res['em_nll']:.3e} (match={res['em_matched']}) | "
              f"SGD: {res['sgd_nll']:.3e} (match={res['sgd_matched']})")
else:
    print("Running Hybrid Optimization + policy-match test... (this may take a while)")
    start_time = time.perf_counter()

    theta_history = results[best_restart]["theta_history"]
    mapso_nll_history = results[best_restart]["gbest_history"]
    mapso_time_history = results[best_restart].get("time_history", None)

    checkpoint_iters = np.unique(np.round(np.geomspace(1, len(theta_history), n_checkpoints)).astype(int) - 1)
    hybrid_results = []
    hybrid_histories = {}

    for it in checkpoint_iters:
        idx_label = list(checkpoint_iters).index(it) + 1
        theta = theta_history[it]
        rho_warm, pi_warm, g_warm = unpack_theta(theta, M, A, Y)

        mapso_step_agree, mapso_traj_agree = replay_policy_match(rho_warm, pi_warm, g_warm, val_set)

        print(f"\n---EM (iter={idx_label}/{len(checkpoint_iters)})---")
        # train_em_with_restarts_parallel returns 8 values in this order
        # (matches the unpacking used for the top-level EM fit above):
        #   rho, pi, g, nll (best restart's FINAL scalar NLL), best_restart
        #   (int index), results, all_nll, all_nll_histories (list of full
        #   per-iteration traces, one per restart). The previous version of
        #   this cell mislabeled the 5th slot (best_restart, an int) as
        #   em_nll_history, so it silently held a scalar instead of a trace
        #   -- fixed by unpacking correctly and indexing into
        #   all_nll_histories with best_restart to get the real trace.
        em_rho, em_pi, em_g, em_nll, em_best_restart_ckpt, em_results_ckpt, em_all_nll_ckpt, em_all_nll_histories_ckpt = train_em_with_restarts_parallel(
            dataset_encoded, M, A, Y, max_iter=2000, n_restarts=n_restart_hybrid, seed=gen_seed,
            warm_params=(rho_warm, pi_warm, g_warm), warm_restarts=None, warm_sigma=0.2, verbose=False, n_jobs=-1
        )
        em_nll_history = np.asarray(em_all_nll_histories_ckpt[em_best_restart_ckpt])
        em_step_agree, em_traj_agree = replay_policy_match(em_rho, em_pi, em_g, val_set)

        print(f"\n---SGD (iter={idx_label}/{len(checkpoint_iters)})---")
        sgd_rho, sgd_pi, sgd_g, sgd_hist, _, _, _, _, _, _ = train_sgd_restarts_parallel(
            dataset_encoded, M, A, Y, n_restarts=n_restart_hybrid, seed=gen_seed, n_epochs=2000,
            warm_params=(rho_warm, pi_warm, g_warm), warm_restarts=None, warm_sigma=0.2,
            optimizer_kind="adam", lr=0.1, batch_size=16, patience=20, dataset_nll_fn=dataset_nll, n_jobs=-1
        )
        sgd_step_agree, sgd_traj_agree = replay_policy_match(sgd_rho, sgd_pi, sgd_g, val_set)

        hybrid_results.append({
            "mapso_iter": int(it),
            "mapso_nll": float(mapso_nll_history[it]),
            "mapso_time": None if mapso_time_history is None else float(mapso_time_history[it]),
            "mapso_matched": bool(mapso_step_agree >= 1.0),
            "mapso_step_agreement": float(mapso_step_agree),
            "mapso_traj_agreement": float(mapso_traj_agree),

            "em_nll": float(em_nll),
            "em_matched": bool(em_step_agree >= 1.0),
            "em_step_agreement": float(em_step_agree),
            "em_traj_agreement": float(em_traj_agree),

            "sgd_nll": float(sgd_hist[-1]),
            "sgd_matched": bool(sgd_step_agree >= 1.0),
            "sgd_step_agreement": float(sgd_step_agree),
            "sgd_traj_agreement": float(sgd_traj_agree),

            "mapso_rho": rho_warm, "mapso_pi": pi_warm, "mapso_g": g_warm,
            "em_rho": em_rho, "em_pi": em_pi, "em_g": em_g,
            "sgd_rho": sgd_rho, "sgd_pi": sgd_pi, "sgd_g": sgd_g,
        })

        # separate, keyed-by-iter cache for the full convergence curves
        hybrid_histories[str(int(it))] = {
            "em_history": np.asarray(em_nll_history).tolist(),
            "sgd_history": np.asarray(sgd_hist).tolist(),
        }

    os.makedirs(os.path.dirname(cache_file), exist_ok=True)

    with open(cache_file, "w") as f:
        json.dump(hybrid_results, f, indent=4, default=numpy_safe)
    with open(history_cache_file, "w") as f:
        json.dump(hybrid_histories, f, default=numpy_safe)

    print(f"Results saved to {cache_file}.")
    print(f"Histories saved to {history_cache_file}.")

### Hybrid results plot

NLL vs. MAPSO handoff iteration for MAPSO, MAPSO→EM, and MAPSO→SGD, with filled markers indicating an exact policy match.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import pandas as pd

df = pd.DataFrame(hybrid_results).sort_values("mapso_iter")

plt.figure(figsize=(6.5, 4.5))

series = [
    ("mapso_nll", "mapso_matched", "o", "C0", "MAPSO"),
    ("em_nll",    "em_matched",    "s", "C1", "MAPSO → EM"),
    ("sgd_nll",   "sgd_matched",   "^", "C2", "MAPSO → SGD"),
]

for nll_col, match_col, marker, color, label in series:
    plt.plot(df["mapso_iter"], df[nll_col], "-", lw=1.5, color=color, alpha=0.5, zorder=1)
    matched = df[match_col].astype(bool)
    plt.scatter(df.loc[matched, "mapso_iter"], df.loc[matched, nll_col],
                marker=marker, s=80, facecolor=color, edgecolor="black", linewidth=0.8, zorder=3)
    plt.scatter(df.loc[~matched, "mapso_iter"], df.loc[~matched, nll_col],
                marker=marker, s=80, facecolor="none", edgecolor=color, linewidth=1.8, zorder=3)

ax = plt.gca()
plt.yscale("symlog", linthresh=1e-4)
ax.yaxis.set_major_formatter(ticker.LogFormatterMathtext())
plt.xlabel("MAPSO iterations before handoff")
plt.ylabel("Final NLL")
plt.title("Hybrid Optimization — NLL vs. exact policy recovery")
plt.grid(True, which="both", alpha=0.3)

method_handles = [Line2D([0], [0], marker=m, color=c, linestyle="", markersize=8,
                          markerfacecolor=c, markeredgecolor="black")
                   for _, _, m, c, _ in series]
method_labels = [lab for *_, lab in series]
style_handles = [
    Line2D([0], [0], marker="o", color="gray", linestyle="", markersize=8,
           markerfacecolor="gray", markeredgecolor="black", label="Policy match"),
    Line2D([0], [0], marker="o", color="gray", linestyle="", markersize=8,
           markerfacecolor="none", markeredgecolor="gray", markeredgewidth=1.8, label="No match"),
]
leg1 = ax.legend(method_handles, method_labels, loc="upper right", fontsize=8, title="Method")
ax.add_artist(leg1)
ax.legend(handles=style_handles, loc="lower left", fontsize=8)

plt.tight_layout()
plt.show()

## 7. Label-switching alignment and comparison tables

FSC latent node indices are arbitrary up to permutation ("label switching"). Each method's learned nodes are aligned to the true memory value `m` they represent via a forward-backward posterior matched against the known true `m` at every timestep, using linear-sum assignment (Hungarian algorithm) on total overlap mass. Aligned parameters are then compared against ground truth.

In [ ]:
def align_to_true_m(dataset_raw, dataset_encoded, rho, pi, g, ms_sorted, m_to_idx):
    M = len(ms_sorted)
    overlap = np.zeros((M, M))          # overlap[learned_k, true_idx]
    for ep, (actions, obs) in zip(dataset_raw, dataset_encoded):
        _, _, gamma, _, _ = forward_backward(actions, obs, rho, pi, g)
        for t, step in enumerate(ep['trajectory']):
            overlap[:, m_to_idx[step['m']]] += gamma[t]
    learned_idx, true_idx = linear_sum_assignment(-overlap)   # maximize overlap
    perm = np.zeros(M, dtype=int)                              # perm[learned_k] = true_idx
    perm[learned_idx] = true_idx
    inv_perm = np.argsort(perm)                                 # inv_perm[true_idx] = learned_k
    rho_a = rho[inv_perm]
    pi_a = pi[:, inv_perm]
    g_a = g[inv_perm][:, inv_perm, :, :]
    return rho_a, pi_a, g_a, overlap


rho_mapso_a, pi_mapso_a, g_mapso_a, ov_mapso = align_to_true_m(
    dataset_raw, dataset_encoded, rho_mapso, pi_mapso, g_mapso, ms_sorted, m_to_idx)
rho_em_a, pi_em_a, g_em_a, ov_em = align_to_true_m(
    dataset_raw, dataset_encoded, rho_em, pi_em, g_em, ms_sorted, m_to_idx)
rho_sgd_a, pi_sgd_a, g_sgd_a, ov_sgd = align_to_true_m(
    dataset_raw, dataset_encoded, rho_sgd, pi_sgd, g_sgd, ms_sorted, m_to_idx)
rho_em_warm_a, pi_em_warm_a, g_em_warm_a, ov_em_warm = align_to_true_m(
    dataset_raw, dataset_encoded, rho_em_warm, pi_em_warm, g_em_warm, ms_sorted, m_to_idx)
rho_sgd_warm_a, pi_sgd_warm_a, g_sgd_warm_a, ov_sgd_warm = align_to_true_m(
    dataset_raw, dataset_encoded, sgd_warm_best_rho, sgd_warm_best_pi, sgd_warm_best_g, ms_sorted, m_to_idx)

action_names_by_idx = {0: "OpenLeft", 1: "Listen", 2: "OpenRight"}

print("=" * 100)
print("rho (initial belief over memory nodes)")
print("=" * 100)
header = (f"{'m':>4} | {'true':>8} | {'MAPSO':>8} | {'EM':>8} | {'SGD':>8} | "
          f"{'EM(warm)':>9} | {'SGD(warm)':>10}")
print(header)
for i, m in enumerate(ms_sorted):
    print(f"{m:>4} | {rho_true[i]:8.4f} | {rho_mapso_a[i]:8.4f} | {rho_em_a[i]:8.4f} | "
          f"{rho_sgd_a[i]:8.4f} | {rho_em_warm_a[i]:9.4f} | {rho_sgd_warm_a[i]:10.4f}")

print("\n" + "=" * 100)
print("pi(action | m)  -- showing the probability mass on the TRUE action")
print("=" * 100)
print(header)
for i, m in enumerate(ms_sorted):
    a_true = fsc[m]['action'] + 1     # -1/0/1 -> 0/1/2
    print(f"{m:>4} | {pi_true[a_true, i]:8.4f} | {pi_mapso_a[a_true, i]:8.4f} | "
          f"{pi_em_a[a_true, i]:8.4f} | {pi_sgd_a[a_true, i]:8.4f} | "
          f"{pi_em_warm_a[a_true, i]:9.4f} | {pi_sgd_warm_a[a_true, i]:10.4f}   "
          f"(true action = {action_names_by_idx[a_true]})")

print("\n" + "=" * 100)
print("g(m' | m, Listen, y)  -- probability mass on the TRUE next node")
print("=" * 100)
print(f"{'m':>4} {'y':>10} | {'true':>8} | {'MAPSO':>8} | {'EM':>8} | {'SGD':>8} | "
      f"{'EM(warm)':>9} | {'SGD(warm)':>10}")
for m in ms_sorted:
    node = fsc[m]
    if node['action'] != 0:
        continue
    i = m_to_idx[m]
    for z in (-1, 1):
        y_idx = (z + 1) // 2
        mp_true_idx = m_to_idx[node['g'][z]]
        obs_name = pomdp.obs_names[z]
        print(f"{m:>4} {obs_name:>10} | {g_true[mp_true_idx, i, 1, y_idx]:8.4f} | "
              f"{g_mapso_a[mp_true_idx, i, 1, y_idx]:8.4f} | "
              f"{g_em_a[mp_true_idx, i, 1, y_idx]:8.4f} | "
              f"{g_sgd_a[mp_true_idx, i, 1, y_idx]:8.4f} | "
              f"{g_em_warm_a[mp_true_idx, i, 1, y_idx]:9.4f} | "
              f"{g_sgd_warm_a[mp_true_idx, i, 1, y_idx]:10.4f}")

print("\n" + "=" * 100)
print("Dataset NLL summary")
print("=" * 100)
print(f"{'true':>10} {'MAPSO':>10} {'EM':>10} {'SGD':>10} {'EM(warm)':>10} {'SGD(warm)':>10}")
print(f"{nll_true:10.3e} {mapso_history[-1]:10.3e} {em_nll_hist[-1]:10.3e} "
      f"{sgd_nll_hist[-1]:10.3e} {em_warm_nll_hist[-1]:10.3e} {sgd_warm_best_nll_history[-1]:10.3e}")

## 8. FSC visualization

Renders the true agent's FSC alongside each method's aligned, learned FSC (MAPSO, EM, SGD, and their MAPSO-warm-started EM/SGD counterparts) using the same diagram style.

In [ ]:
%matplotlib inline
def learned_params_to_fsc(pi_a, g_a, ms_sorted, listen_idx=1):
    learned_fsc = {}
    for m in ms_sorted:
        i = m_to_idx[m]
        action_idx = int(np.argmax(pi_a[:, i]))
        action = action_idx - 1
        if action == 0:
            g_dict, p_dict = {}, {}
            for z in (-1, 1):
                y_idx = (z + 1) // 2
                trans = g_a[:, i, listen_idx, y_idx]
                mp_idx = int(np.argmax(trans))
                g_dict[z] = ms_sorted[mp_idx]
                p_dict[z] = float(trans[mp_idx])
            learned_fsc[m] = {'m': m, 'action': 0, 'g': g_dict, 'p': p_dict}
        else:
            learned_fsc[m] = {'m': m, 'action': action,
                               'g': {1: None, -1: None}, 'p': {1: None, -1: None}}
    return learned_fsc



mapso_fsc = learned_params_to_fsc(pi_mapso_a, g_mapso_a, ms_sorted)
em_fsc = learned_params_to_fsc(pi_em_a, g_em_a, ms_sorted)
sgd_fsc = learned_params_to_fsc(pi_sgd_a, g_sgd_a, ms_sorted)
em_warm_fsc = learned_params_to_fsc(pi_em_warm_a, g_em_warm_a, ms_sorted)
sgd_warm_fsc = learned_params_to_fsc(pi_sgd_warm_a, g_sgd_warm_a, ms_sorted)


def plot_memory_fsc_on_ax(ax, pomdp, fsc, mc, label):
    ms = sorted(fsc.keys())
    node_radius = 0.35

    action_colors = {"Listen": "#a1c9f4", "OpenLeft": "#ffb482", "OpenRight": "#8de5a1"}
    obs_colors = {"HearLeft": "#e74c3c", "HearRight": "#3498db"}
    obs_zorder = {"HearLeft": 1, "HearRight": 2}

    if mc > 0:
        ax.axvspan(-mc + node_radius, mc - node_radius, ymin=0.05, ymax=0.95,
                   color="#a1c9f4", alpha=0.08, zorder=0)
        for boundary in (-mc, mc):
            ax.axvline(boundary, color="gray", linestyle=":", linewidth=1, zorder=0)

    for m in ms:
        node = fsc[m]
        action_name = pomdp.action_names[node['action']]
        color = action_colors.get(action_name, "lightgray")
        beta = np.tanh(pomdp.d * m / 2)

        circle = patches.Circle((m, 0), node_radius, edgecolor='black', facecolor=color, zorder=3)
        ax.add_patch(circle)
        ax.text(m, 0, f"m={m:+d}\n{action_name}", ha='center', va='center',
                 zorder=4, fontsize=8, fontweight='bold')
        ax.text(m, -0.85, f"\u03b2={beta:+.2f}", ha='center', va='top',
                 fontsize=7, color='dimgray', zorder=4)

    BLUE_RAD = -0.5
    RED_RAD = -0.5
    for m in ms:
        node = fsc[m]
        if node['action'] != 0:
            direction = 1 if m >= 0 else -1
            ax.annotate("", xy=(m + 0.55 * direction, 1.9), xytext=(m + 0.05 * direction, node_radius),
                        arrowprops=dict(arrowstyle="-|>", color="gray",
                                         linestyle="dashed", linewidth=1.3), zorder=2)
            ax.text(m + 0.55 * direction, 2.0, "terminal", ha='center', va='bottom',
                     fontsize=7, color="gray", style='italic', zorder=4)
            continue

        for z, m_next in node['g'].items():
            obs_name = pomdp.obs_names[z]
            color = obs_colors.get(obs_name, "black")
            p = node['p'][z]
            rad = BLUE_RAD if z == 1 else RED_RAD

            arc = patches.FancyArrowPatch(
                (m, 0), (m_next, 0),
                connectionstyle=f"arc3,rad={rad}",
                arrowstyle="-|>", mutation_scale=15, color=color, zorder=obs_zorder.get(obs_name, 1),
                shrinkA=22, shrinkB=45, linewidth=1.5, alpha=0.9
            )
            ax.add_patch(arc)

            mid_x = (m + m_next) / 2
            label_y = 0.55 if z == 1 else -0.55

    legend_elements = [
        Line2D([0], [0], color=color, lw=2.5, label=obs_name)
        for obs_name, color in obs_colors.items()
    ] + [
        patches.Patch(facecolor=color, edgecolor='black', label=name)
        for name, color in action_colors.items()
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=7, title_fontsize=8)

    ax.set_xlim(min(ms) - 1, max(ms) + 1)
    ax.set_ylim(-1.6, 2.0)
    ax.axis('off')
    ax.set_title(label, fontsize=11, fontweight='bold', pad=10)


fig, axes = plt.subplots(6, 1, figsize=(max(10, 1.6 * M), 30))
plot_memory_fsc_on_ax(axes[0], pomdp, fsc, mc, f"True agent  (NLL={nll_true:.1e})")
plot_memory_fsc_on_ax(axes[1], pomdp, mapso_fsc, mc, f"MAPSO-fit  (NLL={mapso_history[-1]:.1e})")
plot_memory_fsc_on_ax(axes[2], pomdp, em_fsc, mc, f"EM-fit  (NLL={em_nll_hist[-1]:.1e})")
plot_memory_fsc_on_ax(axes[3], pomdp, sgd_fsc, mc, f"SGD-fit  (NLL={sgd_nll_hist[-1]:.1e})")
plot_memory_fsc_on_ax(axes[4], pomdp, em_warm_fsc, mc, f"EM-fit (MAPSO warm-start)  (NLL={em_warm_nll_hist[-1]:.1e})")
plot_memory_fsc_on_ax(axes[5], pomdp, sgd_warm_fsc, mc, f"SGD-fit (MAPSO warm-start)  (NLL={sgd_warm_best_nll_history[-1]:.1e})")
plt.tight_layout()
#plt.savefig("fsc_comparison.png", dpi=150, bbox_inches='tight')

## 9. Train/val NLL scatter across all trials

Plots per-restart train vs. validation NLL for MAPSO, EM, and SGD to visualize generalization gap across methods.

In [ ]:
fitted_all_trials = {
    "MAPSO": mapso_trials,
    "EM":    em_trials,
    "SGD":   sgd_trials,
}

trial_metrics = evaluate_methods_all_trials(fitted_all_trials, train_set, val_set, per_timestep=False)
methods, train_nll, val_nll, gap = flatten_for_scatter(trial_metrics)

colors = {"MAPSO": "#d62728", "EM": "#1f77b4", "SGD": "#2ca02c"}  # your existing colors dict

fig, axes = plot_train_val_trials(trial_metrics, methods=["MAPSO", "EM", "SGD"],
                                   colors=colors, mc=mc, n_data=n_data)

## 10. Persist results to `results_log.csv`

Logs this session's results (mc, n_episodes, all NLL histories, aligned parameters, overlap matrices, and the hybrid-optimization results) to a running CSV, overwriting any prior row for the same `(mc, n_episodes)`. Storing full histories and aligned parameters lets comparison tables and FSC plots be reconstructed later without rerunning any fitting. Run this cell once at the end of each notebook session/trial.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# --- single source of truth for the NLL normalization convention used
#     throughout this cell. MAPSO/EM/SGD all report per-trajectory NLL
#     internally (nll /= len(dataset)), so per_timestep=False here keeps
#     everything logged in this file on that same convention. ---
PER_TIMESTEP = False

def numpy_safe(obj):
    """default= handler for json.dumps: converts numpy scalars/arrays to
    native Python types so json doesn't choke on int64/float64/etc."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.bool_):
        return bool(obj)
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

def arr_to_json(a):
    """Serialize a numpy array to JSON, preserving shape for reload."""
    a = np.asarray(a)
    return json.dumps({"shape": list(a.shape), "data": a.ravel().tolist()}, default=numpy_safe)


def json_to_arr(s):
    """Inverse of arr_to_json."""
    d = json.loads(s)
    return np.array(d["data"]).reshape(d["shape"])

# --- validation NLLs, computed once here so the plotting notebook just reads them ---
mapso_val_nll    = dataset_nll(val_set, rho_mapso_a, pi_mapso_a, g_mapso_a, per_timestep=PER_TIMESTEP)
em_val_nll       = dataset_nll(val_set, rho_em_a, pi_em_a, g_em_a, per_timestep=PER_TIMESTEP)
em_warm_val_nll  = dataset_nll(val_set, rho_em_warm_a, pi_em_warm_a, g_em_warm_a, per_timestep=PER_TIMESTEP)
sgd_val_nll      = dataset_nll(val_set, rho_sgd_a, pi_sgd_a, g_sgd_a, per_timestep=PER_TIMESTEP)
sgd_warm_val_nll = dataset_nll(val_set, rho_sgd_warm_a, pi_sgd_warm_a, g_sgd_warm_a, per_timestep=PER_TIMESTEP)
nll_true_val     = dataset_nll(val_set, rho_true, pi_true, g_true, per_timestep=PER_TIMESTEP)  # baseline reference

# --- train NLLs, computed the SAME way (raw dataset_nll on train_set) so
#     these match what evaluate_methods()/plot_train_val_bars() actually
#     show -- overwrites the optimizer-tracked loss (which may include a
#     regularization/smoothing term and run slightly higher than raw NLL) ---
mapso_nll     = dataset_nll(train_set, rho_mapso_a, pi_mapso_a, g_mapso_a, per_timestep=PER_TIMESTEP)
em_nll        = dataset_nll(train_set, rho_em_a, pi_em_a, g_em_a, per_timestep=PER_TIMESTEP)
em_warm_nll   = dataset_nll(train_set, rho_em_warm_a, pi_em_warm_a, g_em_warm_a, per_timestep=PER_TIMESTEP)
sgd_nll       = dataset_nll(train_set, rho_sgd_a, pi_sgd_a, g_sgd_a, per_timestep=PER_TIMESTEP)
sgd_warm_nll  = dataset_nll(train_set, rho_sgd_warm_a, pi_sgd_warm_a, g_sgd_warm_a, per_timestep=PER_TIMESTEP)

# --- per-trial (per-restart) train/val NLL, for the jitter/scatter plots --
# (evaluate_methods_all_trials scores EVERY restart, not just the winner,
# so the full spread survives into the log -- rebuildable later without
# rerunning any fitting). Adjust the key names below ('gbest', 'rho'/'pi'/'g')
# if your restart-result dicts use different ones.
mapso_trials = [unpack_theta(r['gbest'], M, A, Y) for r in results]
em_trials    = [(r['rho'], r['pi'], r['g']) for r in em_results]
sgd_trials   = [(r['rho'], r['pi'], r['g']) for r in sgd_results]

fitted_all_trials = {"MAPSO": mapso_trials, "EM": em_trials, "SGD": sgd_trials}
trial_metrics = evaluate_methods_all_trials(fitted_all_trials, train_set, val_set, per_timestep=PER_TIMESTEP)


log_path = Path("results_log.csv")

row = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    # --- key columns: overwrite matches on (mc, n_episodes) ---
    "mc": mc,
    "n_episodes": n_data,

    # --- config ---
    "delta": pomdp.delta,
    "alpha": pomdp.alpha,
    "M": M,
    "ms_sorted": json.dumps(ms_sorted),
    "per_timestep": PER_TIMESTEP,  # record which normalization convention produced this row

    # --- dataset, so a separate plotting-only notebook can rebuild
    #     everything (overlap, gamma, etc.) without regenerating data ---
    "dataset_raw": json.dumps(dataset_raw, default=numpy_safe),
    "dataset_encoded": json.dumps([[list(actions), list(obs)] for actions, obs in dataset_encoded], default=numpy_safe),

    # --- fsc structure needed to rebuild comparison tables / plots ---
    # (action per m, g-transition per (m,z), listen-obs names)
    "fsc_actions": json.dumps({str(m): fsc[m]['action'] for m in ms_sorted}, default=numpy_safe),
    "fsc_g": json.dumps({str(m): {str(z): fsc[m]['g'][z] for z in (-1, 1)}
                          for m in ms_sorted if fsc[m]['action'] == 0}, default=numpy_safe),
    "action_names_by_idx": json.dumps(action_names_by_idx, default=numpy_safe),
    "obs_names": json.dumps({str(k): v for k, v in pomdp.obs_names.items()}, default=numpy_safe),

    # --- ground truth ---
    "nll_true": nll_true,
    "rho_true": arr_to_json(rho_true),
    "pi_true": arr_to_json(pi_true),
    "g_true": arr_to_json(g_true),

    # --- MAPSO ---
    "mapso_nll": mapso_nll,
    "mapso_nll_history": json.dumps(list(mapso_history), default=numpy_safe),
    "mapso_all_nll": json.dumps(list(all_nll), default=numpy_safe),
    "mapso_all_nll_histories": json.dumps([list(h) for h in all_nll_histories], default=numpy_safe),
    "mapso_train_nll_trials": json.dumps(trial_metrics["MAPSO"]["train_nll"], default=numpy_safe),
    "mapso_val_nll_trials": json.dumps(trial_metrics["MAPSO"]["val_nll"], default=numpy_safe),
    "mapso_n_particles": n_particles,
    "mapso_n_iterations": n_iterations,
    "mapso_n_restarts": n_restarts,
    "mapso_best_restart": best_restart,
    "rho_mapso_a": arr_to_json(rho_mapso_a),
    "pi_mapso_a": arr_to_json(pi_mapso_a),
    "g_mapso_a": arr_to_json(g_mapso_a),
    "ov_mapso": arr_to_json(ov_mapso),

    # --- EM ---
    "em_nll": em_nll,
    "em_nll_history": json.dumps(list(em_nll_hist), default=numpy_safe),
    "em_all_nll": json.dumps(list(em_all_nll), default=numpy_safe),
    "em_all_nll_histories": json.dumps([list(h) for h in em_all_nll_histories], default=numpy_safe),
    "em_train_nll_trials": json.dumps(trial_metrics["EM"]["train_nll"], default=numpy_safe),
    "em_val_nll_trials": json.dumps(trial_metrics["EM"]["val_nll"], default=numpy_safe),
    "em_n_restarts": len(em_all_nll),
    "em_best_restart": em_best_restart,
    "rho_em_a": arr_to_json(rho_em_a),
    "pi_em_a": arr_to_json(pi_em_a),
    "g_em_a": arr_to_json(g_em_a),
    "ov_em": arr_to_json(ov_em),

    # --- EM (MAPSO warm-start) ---
    "em_warm_nll": em_warm_nll,
    "em_warm_nll_history": json.dumps(list(em_warm_nll_hist), default=numpy_safe),
    "rho_em_warm_a": arr_to_json(rho_em_warm_a),
    "pi_em_warm_a": arr_to_json(pi_em_warm_a),
    "g_em_warm_a": arr_to_json(g_em_warm_a),
    "ov_em_warm": arr_to_json(ov_em_warm),

    # --- SGD ---
    "sgd_nll": sgd_nll,
    "sgd_nll_history": json.dumps(list(sgd_nll_hist), default=numpy_safe),
    "sgd_all_nll": json.dumps(list(sgd_all_nll), default=numpy_safe),
    "sgd_all_nll_histories": json.dumps([list(h) for h in sgd_all_nll_histories], default=numpy_safe),
    "sgd_train_nll_trials": json.dumps(trial_metrics["SGD"]["train_nll"], default=numpy_safe),
    "sgd_val_nll_trials": json.dumps(trial_metrics["SGD"]["val_nll"], default=numpy_safe),
    "sgd_n_restarts": len(sgd_all_nll),
    "sgd_best_restart": sgd_best_restart,
    "rho_sgd_a": arr_to_json(rho_sgd_a),
    "pi_sgd_a": arr_to_json(pi_sgd_a),
    "g_sgd_a": arr_to_json(g_sgd_a),
    "ov_sgd": arr_to_json(ov_sgd),

    # --- SGD (MAPSO warm-start) ---
    "sgd_warm_nll": sgd_warm_nll,
    "sgd_warm_nll_history": json.dumps(list(sgd_warm_best_nll_history), default=numpy_safe),
    "sgd_warm_all_nll": json.dumps(list(sgd_warm_all_nll), default=numpy_safe),
    "sgd_warm_all_nll_histories": json.dumps([list(h) for h in sgd_warm_all_nll_histories], default=numpy_safe),
    "sgd_warm_n_restarts": len(sgd_warm_all_nll),
    "sgd_warm_best_restart": sgd_warm_best_restart,
    "rho_sgd_warm_a": arr_to_json(rho_sgd_warm_a),
    "pi_sgd_warm_a": arr_to_json(pi_sgd_warm_a),
    "g_sgd_warm_a": arr_to_json(g_sgd_warm_a),
    "ov_sgd_warm": arr_to_json(ov_sgd_warm),


    # --- runtime ---
    "mapso_runtime_sec": mapso_runtime,
    "em_runtime_sec": em_runtime,
    "em_warm_runtime_sec": em_warm_runtime,
    "sgd_runtime_sec": sgd_runtime,
    "sgd_warm_runtime_sec": sgd_warm_runtime,

    # --- validation split metadata ---
    "n_val": len(val_set),
    "val_split_seed": 2,   # whatever seed you passed to split_dataset_n

    # --- held-out data itself, same serialization pattern as dataset_encoded ---
    "dataset_val_encoded": json.dumps(
        [[list(actions), list(obs)] for actions, obs in val_set], default=numpy_safe
    ),

    # --- validation NLLs per method (this is what the plotting notebook actually plots) ---
    "nll_true_val": nll_true_val,
    "mapso_val_nll": mapso_val_nll,
    "em_val_nll": em_val_nll,
    "em_warm_val_nll": em_warm_val_nll,
    "sgd_val_nll": sgd_val_nll,
    "sgd_warm_val_nll": sgd_warm_val_nll,

    "notes": "",   # fill in before running -- what changed this trial

    # --- Hybrid MAPSO -> local optimizer experiment ---
    "hybrid_results": json.dumps(hybrid_results, default=numpy_safe),
    "hybrid_mapso_iters": json.dumps([int(r["mapso_iter"]) for r in hybrid_results], default=numpy_safe),
    "hybrid_mapso_nll": json.dumps([float(r["mapso_nll"]) for r in hybrid_results], default=numpy_safe),
    "hybrid_em_nll": json.dumps([float(r["em_nll"]) for r in hybrid_results], default=numpy_safe),
    "hybrid_sgd_nll": json.dumps([float(r["sgd_nll"]) for r in hybrid_results], default=numpy_safe),


}

# --- load existing log (if any), drop any row matching this (mc, n_episodes), append new row ---
if log_path.exists():
    log = pd.read_csv(log_path)
    log = log[~((log["mc"] == mc) & (log["n_episodes"] == n_data))]
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
else:
    log = pd.DataFrame([row])

log.to_csv(log_path, index=False)

print(f"Logged trial (mc={mc}, n_episodes={n_data}) to {log_path.resolve()}")
print(f"Total distinct (mc, n_episodes) trials in log: {len(log)}")
print(f"NLL normalization used for this row: per_timestep={PER_TIMESTEP}")

## Utility: remove a trial from the results log

Optional maintenance snippet for dropping a specific `(mc, n_episodes)` row from `results_log.csv`. Left commented out since it mutates the log file on disk — uncomment and edit the filter before running.

In [ ]:
#Remove unwanted trials

# import pandas as pd

# log_path = "results_log.csv"
# log = pd.read_csv(log_path)

# before = len(log)
# log = log[~((log["mc"] == 4) & (log["n_episodes"] == 30))]
# after = len(log)

# log.to_csv(log_path, index=False)
# print(f"Removed {before - after} row(s). {after} rows remain.")